In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from pyspark.sql import SparkSession, Row
from delta import *
from delta.tables import DeltaTable

_nb_file = globals().get('__vsc_ipynb_file__', '')
_project_root = Path(_nb_file).parent.parent if _nb_file else Path(__file__).parent.parent
load_dotenv(_project_root / '.env', override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

# Para o SparkSession existente para limpar o cache interno do DeltaLog
active = SparkSession.getActiveSession()
if active:
    active.stop()
    print('SparkSession anterior encerrada (cache limpo).')

spark = (
    SparkSession.builder
    .appName('DML_Delta_Lake')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)

def dp(tabela):
    return f's3a://{BRONZE_BUCKET}/{tabela}'

def dt(tabela):
    return DeltaTable.forPath(spark, dp(tabela))

def dr(tabela):
    return spark.read.format('delta').load(dp(tabela))

print('SparkSession criada com sucesso!')
print(f'MinIO: {MINIO_ENDPOINT} | Bronze: {BRONZE_BUCKET}')
spark

your 131072x1 screen size is bogus. expect trouble
26/05/04 21:54:03 WARN Utils: Your hostname, DESKTOP-S3JB25M resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/04 21:54:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/c/Users/felip/www/university/eng-dados/projeto-2/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/felipe/.ivy2/cache
The jars for the packages stored in: /home/felipe/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5ff51866-b502-4b27-8107-ac91cc2b2f09;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 465ms :: artifacts dl 27ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hado

SparkSession criada com sucesso!
MinIO: http://localhost:9020 | Bronze: bronze


In [2]:
tabelas_delta = ['albuns', 'artistas', 'musicas', 'reproducoes', 'usuarios']

print(f'{"Tabela":<15} {"Registros":>10}')
print('-' * 27)
for tabela in tabelas_delta:
    count = dr(tabela).count()
    print(f'{tabela:<15} {count:>10}')

Tabela           Registros
---------------------------


26/05/04 21:54:21 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/04 21:54:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


albuns                   5


artistas                 5
musicas                  5
reproducoes              5
usuarios                 5


In [3]:
print('=== ARTISTAS ===')
dr('artistas').orderBy('id').show()

print('=== USUARIOS ===')
dr('usuarios').orderBy('id').show()

print('=== MUSICAS ===')
dr('musicas').orderBy('id').show()

=== ARTISTAS ===


+---+--------------+--------------+-----------+--------------------+
| id|          nome|          pais|     genero|           criado_em|
+---+--------------+--------------+-----------+--------------------+
|  1|    The Weeknd|        Canadá|        R&B|2026-05-05 00:26:...|
|  2|      Dua Lipa|   Reino Unido|        Pop|2026-05-05 00:26:...|
|  3|Kendrick Lamar|Estados Unidos|    Hip-Hop|2026-05-05 00:26:...|
|  4| Billie Eilish|Estados Unidos|  Indie Pop|2026-05-05 00:26:...|
|  5|   Tame Impala|     Austrália|Psicodélico|2026-05-05 00:26:...|
+---+--------------+--------------+-----------+--------------------+

=== USUARIOS ===
+---+-------------+--------------------+-------+--------------------+
| id|         nome|               email|  plano|           criado_em|
+---+-------------+--------------------+-------+--------------------+
|  1|     Ana Lima|  ana.lima@email.com|premium|2026-05-05 00:26:...|
|  2|  Bruno Costa|bruno.costa@email...|   free|2026-05-05 00:26:...|
|  3|  Carl

## INSERT — novo artista e novo usuário

In [6]:
from datetime import datetime

artistas_schema = dr('artistas').schema
usuarios_schema = dr('usuarios').schema

novo_artista = spark.createDataFrame(
    [(6, 'Novo Artista', 'Brasil', 'MPB', datetime(2024, 1, 1))],
    schema=artistas_schema
)
novo_artista.write.format('delta').mode('append').save(dp('artistas'))

novo_usuario = spark.createDataFrame(
    [(6, 'Felipe Teste', 'felipe@teste.com', 'premium', datetime(2024, 5, 1))],
    schema=usuarios_schema
)
novo_usuario.write.format('delta').mode('append').save(dp('usuarios'))

print('Após INSERT:')
dr('artistas').orderBy('id').show()
dr('usuarios').orderBy('id').show()

Após INSERT:
+---+--------------+--------------+-----------+--------------------+
| id|          nome|          pais|     genero|           criado_em|
+---+--------------+--------------+-----------+--------------------+
|  1|    The Weeknd|        Canadá|        R&B|2026-05-05 00:26:...|
|  2|      Dua Lipa|   Reino Unido|        Pop|2026-05-05 00:26:...|
|  3|Kendrick Lamar|Estados Unidos|    Hip-Hop|2026-05-05 00:26:...|
|  4| Billie Eilish|Estados Unidos|  Indie Pop|2026-05-05 00:26:...|
|  5|   Tame Impala|     Austrália|Psicodélico|2026-05-05 00:26:...|
|  6|  Novo Artista|        Brasil|        MPB| 2024-01-01 00:00:00|
+---+--------------+--------------+-----------+--------------------+

+---+-------------+--------------------+-------+--------------------+
| id|         nome|               email|  plano|           criado_em|
+---+-------------+--------------------+-------+--------------------+
|  1|     Ana Lima|  ana.lima@email.com|premium|2026-05-05 00:26:...|
|  2|  Bruno Cos

## UPDATE — alterar plano de usuário

In [7]:
dt('usuarios').update(
    condition="plano = 'gratuito'",
    set={"plano": "'basico'"}
)

print('Após UPDATE (gratuito → basico):')
dr('usuarios').select('id', 'nome', 'plano').orderBy('id').show()

Após UPDATE (gratuito → basico):
+---+-------------+-------+
| id|         nome|  plano|
+---+-------------+-------+
|  1|     Ana Lima|premium|
|  2|  Bruno Costa|   free|
|  3|  Carla Souza|premium|
|  4|Diego Martins|   free|
|  5| Elena Ferraz|premium|
|  6| Felipe Teste|premium|
+---+-------------+-------+



## DELETE — remover artista inserido

In [8]:
dt('artistas').delete("id = 6")

print('Após DELETE (artista id=6):')
dr('artistas').orderBy('id').show()

Após DELETE (artista id=6):
+---+--------------+--------------+-----------+--------------------+
| id|          nome|          pais|     genero|           criado_em|
+---+--------------+--------------+-----------+--------------------+
|  1|    The Weeknd|        Canadá|        R&B|2026-05-05 00:26:...|
|  2|      Dua Lipa|   Reino Unido|        Pop|2026-05-05 00:26:...|
|  3|Kendrick Lamar|Estados Unidos|    Hip-Hop|2026-05-05 00:26:...|
|  4| Billie Eilish|Estados Unidos|  Indie Pop|2026-05-05 00:26:...|
|  5|   Tame Impala|     Austrália|Psicodélico|2026-05-05 00:26:...|
+---+--------------+--------------+-----------+--------------------+



## MERGE (UPSERT) — atualizar ou inserir músicas

In [9]:
atualizacoes = spark.createDataFrame([
    Row(id=1, album_id=1, titulo='Música Atualizada', duracao_segundos=999, numero_faixa=1),
    Row(id=99, album_id=2, titulo='Música Nova', duracao_segundos=180, numero_faixa=10),
])

(
    dt('musicas')
    .alias('alvo')
    .merge(atualizacoes.alias('src'), 'alvo.id = src.id')
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print('Após MERGE:')
dr('musicas').orderBy('id').show()

Após MERGE:
+---+--------+--------------------+----------------+------------+
| id|album_id|              titulo|duracao_segundos|numero_faixa|
+---+--------+--------------------+----------------+------------+
|  1|       1|   Música Atualizada|             999|           1|
|  2|       2|          Levitating|             203|           5|
|  3|       3|             Alright|             219|           9|
|  4|       4|   Happier Than Ever|             298|          11|
|  5|       5|The Less I Know t...|             216|          11|
| 99|       2|         Música Nova|             180|          10|
+---+--------+--------------------+----------------+------------+



## Time Travel — histórico de versões

In [10]:
print('=== HISTÓRICO (musicas) ===')
dt('musicas').history().select(
    'version', 'timestamp', 'operation', 'operationParameters'
).show(truncate=False)

print('=== VERSÃO 0 (estado original) ===')
spark.read.format('delta').option('versionAsOf', 0).load(dp('musicas')).show()

=== HISTÓRICO (musicas) ===
+-------+-------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                                                                                                                                                           |
+-------+-------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2      |2026-05-04 21:57:10|MERGE    |{predicate -> ["(cast(id#7976 as bigint) = id#7966L)"], matchedPredicates -> [{"actionType":"update"}], notMatchedPredicates -> [{"actionType":"insert"}], notMatchedBySourcePredicates -> []}|
|1      |2026-05-04 21:49:40|WRITE    |{mode -> 